# Agentic Orchestrator — Interactive Walkthrough

This notebook demonstrates the core capabilities of the Agentic Orchestrator:
1. Goal submission and planning
2. Task DAG visualization
3. Deterministic simulation
4. Audit trail inspection
5. Learned policy training

In [ ]:
import sys
sys.path.insert(0, '..')

from src.agents.planner import PlannerAgent
from src.agents.retriever import RetrieverAgent
from src.agents.executor import ExecutorAgent
from src.agents.verifier import VerifierAgent
from src.audit.logger import AuditLogger
from src.core.models import GoalCreate, GoalConstraints, Goal, GoalStatus
from src.core.orchestrator import Orchestrator
from src.simulator.engine import Simulator, SimulatorConfig

## 1. Set Up Components

In [ ]:
audit = AuditLogger()
planner = PlannerAgent()
retriever = RetrieverAgent()
executor = ExecutorAgent(simulate=True)
verifier = VerifierAgent()
orch = Orchestrator(planner, retriever, executor, verifier, audit)
print('Components initialized.')

## 2. Submit a Campaign Goal

In [ ]:
goal_create = GoalCreate(
    title='Q2 Email Engagement Campaign',
    description='Increase email open rates by 20% for enterprise segment',
    constraints=GoalConstraints(
        budget_usd=5000,
        deadline='2026-06-30',
        channels=['email', 'in-app'],
        audience='enterprise',
    ),
)

goal = await orch.submit_goal(goal_create)
print(f'Goal ID: {goal.id}')
print(f'Status: {goal.status.value}')

## 3. Plan and Execute

In [ ]:
result = await orch.plan_and_execute(goal.id)
print(f'Final Status: {result.status.value}')
print(f'Task Count: {result.task_count}')

## 4. Inspect Task Graph

In [ ]:
graph = orch.get_graph(goal.id)
for task in graph.tasks.values():
    icon = {'completed': '✓', 'failed': '✗', 'skipped': '⏭'}.get(task.status.value, '○')
    print(f'{icon} {task.name} [{task.agent_type.value}] → {task.status.value}')

## 5. View Audit Trail

In [ ]:
events = await audit.get_events(goal_id=goal.id)
for e in sorted(events, key=lambda x: x.timestamp):
    print(f'{e.timestamp.strftime("%H:%M:%S")} | {e.agent:>12} | {e.action}')

## 6. Run Deterministic Simulation

In [ ]:
sim_goal = Goal(
    title='Simulation Test',
    description='Test deterministic simulation',
    constraints=GoalConstraints(channels=['email']),
    status=GoalStatus.PENDING,
)

sim_graph = planner._plan_heuristic(sim_goal)
sim = Simulator(SimulatorConfig(seed=42, failure_rate=0.1))
sim.reset(sim_goal, sim_graph)
state = sim.run_to_completion()

print(f'Steps: {state.step}')
print(f'Total Reward: {state.total_reward:.2f}')
print(f'\nTrace:')
for t in state.trace:
    print(f'  Step {t["step"]}: {t["task_name"]} → {t["status"]} (reward: {t["reward"]:.1f})')

## 7. Generate Synthetic Data

In [ ]:
from data.generator import GeneratorConfig, generate_goals, generate_memory_docs

config = GeneratorConfig(num_goals=10, num_memory_docs=5, seed=42)
goals = generate_goals(config)
docs = generate_memory_docs(config)

print(f'Generated {len(goals)} goals and {len(docs)} memory docs')
print(f'\nSample goal: {goals[0]["title"]}')
print(f'Description: {goals[0]["description"]}')